In [ ]:
# =============================================================================
# USER CONFIG: ??????? ?????? ?????? ??? ??????
# =============================================================================

# SAMPLE_START: ?????? ?????? ?????? ?? test.csv.
# ??????: SAMPLE_START = 0 ????????? ? ?????? ??????.
# ??????: SAMPLE_START = 100 ????????? ?? 101-? ?????? test.csv.
SAMPLE_START = 0

# SAMPLE_COUNT: ??????? ????? test.csv ??????, ??????? ? SAMPLE_START.
# ??????: SAMPLE_COUNT = 1 ??? ???????? smoke-run.
# ??????: SAMPLE_COUNT = 1001 ??? ??????? test.csv.
SAMPLE_COUNT = 1

# GLOBAL_BEAM_WIDTH: ?????????? ?????? ???? ?? ??? GPU ????????.
# ?????? ???????? ?????? ???? ????? ????????, ?? ??????? ?????? ??????? ? static capacity.
# ???????: 2**12, 2**14, 2**16, 2**18.
GLOBAL_BEAM_WIDTH = 2**12

# B_MICRO: ?????? microbatch ??? inference lane.
# ?????? 8192 ?????? ???????? ?? 2xT4; ?????????? ??? ?????????? ?? memory pressure.
B_MICRO = 8192

# BETA: ????????? static capacity ??? next_state_pool/hash buffers.
# ??? ????????? beam ?????? ????? ??????? BETA, ?????? ??? ????? ?????????? ?? prune ????? ???? ?????? ?????? beam.
# ???????: 1.20 ??? beam=2**18; 32.0 ??? 2**12..2**16 smoke/production safety ?? T4.
BETA = 32.0


In [ ]:
# =============================================================================
# ADVANCED CONFIG: ?????? ????; ?????? ?????? ?????? ? env ????? ???????? solver
# =============================================================================

# MAX_DEPTH: ???????????? ??????? beam search.
# ??????: 100 = ??????? ??????????? ?????.
MAX_DEPTH = 100

# INFERENCE_PARALLELISM: ????? CUDA inference lanes ??? Stream1.
# ? ??????? ??????????? lanes ?????????? ???? ??????????? TorchScript scorer ?? rank.
# ??????: 1 = ??????????? ??????; 2 = ?????? ??????? ?? T4.
INFERENCE_PARALLELISM = 2

# K_EXPAND_TILE: ?????? tile ??? Stream2 expand/dedup/prune.
# ??????: 16384 = ??????????? T4 ???????; 8192 = ????? ?????? threshold/prune ?????.
K_EXPAND_TILE = 16384

# SCORE_RING_DEPTH: ??????? ?????? score buffer ????? inference ? expand.
# ??????: 8 = ??????????? T4 ???????.
SCORE_RING_DEPTH = 8

# NET_RING_DEPTH: ??????? ?????? NCCL send/recv buffers ??? Stream3.
# ??????: 2 = ??????????? T4 ???????.
NET_RING_DEPTH = 2

# BUCKET_CAP_PER_PEER: fixed capacity ??? candidate exchange ?? peer ?? tile.
# ??????: 65536 = ??????????? ??????? ??? bucket overflow ? ??????? ??????.
BUCKET_CAP_PER_PEER = 65536

# HASH_LOAD_FACTOR: ??? ?????? ????????, ??? ?????? hash table ? ?????? probe pressure.
# ??????: 0.35 = safety ??? small-beam tests; 0.45 = ????? ?????????? ???????.
HASH_LOAD_FACTOR = 0.35

# PROBE_LIMIT: ???????? linear probes ? GPU hash table.
# ??????: 512 = safety; 256 = ???????/??????????, ?? ???? overflow ????.
PROBE_LIMIT = 512

# HISTORY_BACKEND:
# "cpu" = GPU ?????? ?????? ???? ???? transitions, CPU RAM ?????? archive/history.
# "gpu" = ?????? ?????, GPU ?????? full-depth transition history.
HISTORY_BACKEND = "cpu"

# CPU_HISTORY_CHECKPOINT:
# 0 = ?? ?????? checkpoint ?????? depth.
# 1 = ?????? CPU archive/frontier checkpoint ?? ???? ????? depth; ?????????, ?? resume-safe.
CPU_HISTORY_CHECKPOINT = 0

# RESUME_BEAMSEARCH:
# 0 = ?????????? sample ??????.
# 1 = ?????? latest complete CPU-history checkpoint ? ?????????? ? depth+1.
RESUME_BEAMSEARCH = 0

# RESUME_SUBMISSION:
# 0 = ???????????? /kaggle/working/submission.csv.
# 1 = ?? ?????? initial_state_id, ??? ?????????????? ? submission.csv.
RESUME_SUBMISSION = 0

# CPU_HISTORY_WORKERS: ????? CPU threads ??? reconstruct/write archive ??? checkpoint mode.
CPU_HISTORY_WORKERS = 2

# LOG_EVERY: ???????? SAMPLE_RESULT ?????? N samples; 1 = ?????? sample.
LOG_EVERY = 1

# DEPTH_LOG_EVERY: ???????? DEPTH_RESULT ?????? N depths; 0 = ?????????.
DEPTH_LOG_EVERY = 0

# TIMEOUT_SEC: soft-timeout wrapper ??? ?????? ??????? solver; None = ??? timeout.
# ??????: TIMEOUT_SEC = 1200 ??? 20 ?????.
TIMEOUT_SEC = None

# GITHUB_REPO_URL: ???? C++/CUDA/Python ??? ??????? ??????, ? ?? ?? Kaggle dataset.
# ??????? ????? ?????? ??? ?????? FullBeamNice/weights ??? ???????????????? ??????.
GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_BRANCH = "master"

# PROJECT_DIR: ???? ??????????? ?????? ?????? Kaggle runtime.
PROJECT_DIR = "/kaggle/working/CayleyBeam100H100"

# MODEL_DATASET_HINT: ????? ????? attached dataset ? ???????; ?????? ?????? = ????-?????.
MODEL_DATASET_HINT = "cayleybeam-fullbeamnice-project"

# SCORER_INIT_PY: ???? ? ????????????????? Python initializer; ?????? ?????? = FullBeamNice default model.
# ??????: SCORER_INIT_PY = "/kaggle/working/my_scorer_init.py"
SCORER_INIT_PY = ""

# SUBMISSION_PATH: ????, ??????? ??????? ?? ???? run ? ??????? ???????????? ? submit cell.
SUBMISSION_PATH = "/kaggle/working/submission.csv"


In [ ]:
# =============================================================================
# CUSTOM SCORER API: ??? ?????????? ???? ????????/?????????
# =============================================================================

# Solver ?????? ?????????? scores ????? [B, 24] ? ??????? ??????? actions:
# [-B, -BL, -BR, -D, -DL, -DR, -F, -FL, -FR, -L, -R, -U,
#   B,  BL,  BR,  D,  DL,  DR,  F,  FL,  FR,  L,  R,  U]
# ?????? score = ???????? ?????. Export adapter ???????? output initializer-? ? ????? ?????????.

# ??? ??????????:
# 1. ???????? ????, ???????? /kaggle/working/my_scorer_init.py.
# 2. ? ????? ?????????? create_scorer() ??? build_scorer() ??? MODEL.
# 3. ??????? torch.nn.Module ? metadata dict.
# 4. ??????? SCORER_INIT_PY = "/kaggle/working/my_scorer_init.py" ?? ?????? config cell.

# ?????????????? output_kind:
# output_kind="action24": ?????? ????? ?????????? [B,24]. ????? ?????? action_names ??? remap.
# output_kind="action12": ?????? ?????????? 12 named moves; adapter ???????? named outputs ? 24 slots, ????????????? slots ??????? 0.
# output_kind="value1_after_move": adapter ??? ???????? 24 permutations ? state, ???????? child states ????? value-model [B*24,1], ????? reshape -> [B,24].
# output_kind="heuristic24": TorchScriptable ????????? ?????????? [B,24], ???????? hamming/Bellman-like estimator.

# ??????????? ?????? initializer-?:
# import torch
# from torch import nn
#
# class MyScorer(nn.Module):
#     def forward(self, states_u8: torch.Tensor) -> torch.Tensor:
#         # states_u8: uint8 tensor [B,120]
#         # return: float/int tensor [B,24], higher is better before quantization
#         return torch.zeros((states_u8.shape[0], 24), device=states_u8.device)
#
# def create_scorer():
#     model = MyScorer().eval()
#     meta = {
#         "output_kind": "action24",
#         "higher_is_better": True,
#         "score_scale": 1.0,
#         "score_bias": 0.0,
#     }
#     return model, meta

# ?????? export_fullbeamnice_scorer.py adapter ?????? canonical TorchScript scorer:
# initializer -> raw model output -> optional action-name remap -> optional 24 child-state evaluation -> [B,24] -> int16 score ring.
# C++ hot path ?? ????? ??? 1/12/24 outputs; C++ ?????? ???????? [micro_size,24], ??????? kernels ???????? ????????.

# TODO: generalized_generator_count.
# ?????? CUDA kernels ? action table ????????????? ?? FANOUT=24 ? STATE_SIZE=120.
# ????????? ???: ??????? FANOUT runtime/compile variant, ????? solver ??????????? ????? ????? generators, ? ?? ?????? 24.


In [ ]:
# =============================================================================
# YANDEX CLOUD TODO: ??????? ?????? Kaggle -> Yandex Cloud
# =============================================================================

# ??????? ?????: ??? ?????????? ??????????? ?????? Kaggle runtime, ??? Yandex Cloud.
# TODO: ???????? sync checkpoints/submission/logs ? Yandex Object Storage.
# TODO: ???????? credentials ????? Kaggle Secrets, ?? ????? ???????? notebook code.
# TODO: ???????? resume flow: Kaggle downloads latest checkpoint from Yandex Cloud before run.
# TODO: ???????? upload flow: after each sample/depth sync CPU-history checkpoint and submission.csv.
# TODO: ???????? cluster handoff: Kaggle smoke-run -> Yandex/H100 production run.

print("YANDEX_CLOUD_STATUS: disabled; current_run_location=Kaggle_only")


In [ ]:
# =============================================================================
# RUN SOLVER: clone code from GitHub, copy competition input files, use model dataset only for weights
# =============================================================================

import csv
import json
import os
import pathlib
import queue
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime

KAGGLE_INPUT = pathlib.Path('/kaggle/input')
PROJECT_PATH = pathlib.Path(PROJECT_DIR)


def now():
    return datetime.now().strftime('%H:%M:%S')


def gpu_snapshot():
    try:
        out = subprocess.check_output([
            'nvidia-smi', '--query-gpu=index,name,memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'
        ], text=True, stderr=subprocess.STDOUT).strip()
        return out.replace('\n', ' | ')
    except Exception as exc:
        return f'nvidia-smi-unavailable:{type(exc).__name__}:{exc}'


def run_live(cmd, *, env=None, cwd=None, title='', timeout_sec=None, heartbeat_sec=30):
    print('\n' + '=' * 96, flush=True)
    print(f'[{now()}] {title}', flush=True)
    print('=' * 96, flush=True)
    print(f'[{now()}] cmd={cmd}', flush=True)
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    merged.setdefault('PYTHONUNBUFFERED', '1')
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    q = queue.Queue()

    def reader():
        for line in iter(proc.stdout.readline, ''):
            q.put(line.rstrip('\n'))
        proc.stdout.close()

    threading.Thread(target=reader, daemon=True).start()
    start = time.time()
    last_heartbeat = 0.0
    last_output = time.time()
    lines = []
    while True:
        drained = False
        while True:
            try:
                line = q.get_nowait()
            except queue.Empty:
                break
            drained = True
            last_output = time.time()
            lines.append(line)
            print(line, flush=True)
        rc = proc.poll()
        elapsed = time.time() - start
        if timeout_sec is not None and elapsed > timeout_sec and rc is None:
            proc.kill()
            raise TimeoutError(f'timeout after {elapsed:.1f}s: {cmd}')
        if rc is not None:
            while True:
                try:
                    line = q.get_nowait()
                except queue.Empty:
                    break
                lines.append(line)
                print(line, flush=True)
            print(f'[{now()}] process_exit | returncode={rc} | elapsed_sec={elapsed:.1f}', flush=True)
            text = '\n'.join(lines)
            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd, output=text)
            return text
        if time.time() - last_heartbeat >= heartbeat_sec:
            print(f'[{now()}] still_running | elapsed_sec={elapsed:.1f} | silent_for_sec={time.time()-last_output:.1f} | gpu=[{gpu_snapshot()}]', flush=True)
            last_heartbeat = time.time()
        time.sleep(0.2 if drained else 0.5)


def find_competition_dir():
    candidates = []
    required = ['puzzle_info.json', 'sample_submission.csv', 'test.csv']
    for p in KAGGLE_INPUT.rglob('*'):
        if p.is_dir() and all((p / name).exists() for name in required):
            candidates.append(p)
    if not candidates:
        listing = [str(x) for x in KAGGLE_INPUT.glob('*')]
        raise FileNotFoundError(f'competition files not found under /kaggle/input; top_level={listing}')
    candidates.sort(key=lambda x: (0 if 'competition' in str(x).lower() else 1, len(str(x))))
    return candidates[0]


def find_model_source():
    candidates = []
    for p in KAGGLE_INPUT.rglob('*'):
        if not p.is_dir():
            continue
        has_model = (p / 'FullBeamNice').exists() or (p / 'FullBeamNice.zip').exists()
        hint_ok = not MODEL_DATASET_HINT or MODEL_DATASET_HINT.lower() in str(p).lower()
        if has_model and hint_ok:
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError('model dataset with FullBeamNice or FullBeamNice.zip not found; attach model dataset')
    candidates.sort(key=lambda x: len(str(x)))
    return candidates[0]


def prepare_project_from_github():
    if PROJECT_PATH.exists():
        shutil.rmtree(PROJECT_PATH)
    run_live(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_PATH)], title='clone project from GitHub', timeout_sec=600)

    data_dir = PROJECT_PATH / 'data'
    data_dir.mkdir(parents=True, exist_ok=True)
    competition_dir = find_competition_dir()
    for name in ['puzzle_info.json', 'sample_submission.csv', 'test.csv']:
        src = competition_dir / name
        if not src.exists():
            raise FileNotFoundError(f'competition file missing: {src}')
        shutil.copy2(src, data_dir / name)

    model_src = find_model_source()
    if (model_src / 'FullBeamNice').exists():
        shutil.copytree(model_src / 'FullBeamNice', PROJECT_PATH / 'FullBeamNice', dirs_exist_ok=True)
    elif (model_src / 'FullBeamNice.zip').exists():
        shutil.unpack_archive(str(model_src / 'FullBeamNice.zip'), PROJECT_PATH / 'FullBeamNice')
    else:
        raise FileNotFoundError(f'FullBeamNice model source missing inside {model_src}')

    print('PROJECT_SOURCE_GITHUB', GITHUB_REPO_URL, GITHUB_BRANCH, flush=True)
    print('COMPETITION_DATA_DIR', str(competition_dir), flush=True)
    print('MODEL_SOURCE_DIR', str(model_src), flush=True)
    print('PROJECT_READY', str(PROJECT_PATH), flush=True)


prepare_project_from_github()

base_env = {
    'PYTHONUNBUFFERED': '1',
    'CUDA_VISIBLE_DEVICES': '0,1',
    'USE_CUDA_GRAPHS': '1',
    'INFERENCE_BACKEND': 'torchscript_ensemble',
    'INFERENCE_PARALLELISM': INFERENCE_PARALLELISM,
    'HISTORY_BACKEND': HISTORY_BACKEND,
    'CPU_HISTORY_CHECKPOINT': CPU_HISTORY_CHECKPOINT,
    'RESUME_BEAMSEARCH': RESUME_BEAMSEARCH,
    'RESUME_SUBMISSION': RESUME_SUBMISSION,
    'CPU_HISTORY_WORKERS': CPU_HISTORY_WORKERS,
    'GLOBAL_BEAM_WIDTH': GLOBAL_BEAM_WIDTH,
    'MAX_DEPTH': MAX_DEPTH,
    'B_MICRO': B_MICRO,
    'K_EXPAND_TILE': K_EXPAND_TILE,
    'SCORE_RING_DEPTH': SCORE_RING_DEPTH,
    'NET_RING_DEPTH': NET_RING_DEPTH,
    'BUCKET_CAP_PER_PEER': BUCKET_CAP_PER_PEER,
    'BETA': BETA,
    'HASH_LOAD_FACTOR': HASH_LOAD_FACTOR,
    'PROBE_LIMIT': PROBE_LIMIT,
    'TEST_START': SAMPLE_START,
    'TEST_COUNT': SAMPLE_COUNT,
    'SUBMISSION_PATH': SUBMISSION_PATH,
    'SUBMISSION_APPEND_EACH': '1',
    'LOG_EVERY': LOG_EVERY,
    'DEPTH_LOG_EVERY': DEPTH_LOG_EVERY,
    'NCCL_IB_DISABLE': '1',
    'NCCL_P2P_DISABLE': '1',
    'NCCL_SOCKET_IFNAME': 'lo',
    'GLOO_SOCKET_IFNAME': 'lo',
    'NCCL_DEBUG': 'WARN',
    'BUILD_VERBOSE': '0',
}
if SCORER_INIT_PY:
    base_env['SCORER_INIT_PY'] = SCORER_INIT_PY

run_live([sys.executable, '-u', 'scripts/t4_sizing.py'], env=base_env, cwd=PROJECT_PATH, title='static memory sizing', timeout_sec=120)
solver_output = run_live([
    sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nnodes=1', '--nproc_per_node=2', 'scripts/solve_testcsv_2gpu.py'
], env=base_env, cwd=PROJECT_PATH, title='2xT4 beam search', timeout_sec=TIMEOUT_SEC)

print('RUN_DONE', json.dumps({
    'submission_path': SUBMISSION_PATH,
    'sample_start': SAMPLE_START,
    'sample_count': SAMPLE_COUNT,
    'global_beam_width': GLOBAL_BEAM_WIDTH,
    'history_backend': HISTORY_BACKEND,
}, ensure_ascii=False), flush=True)


In [ ]:
# =============================================================================
# METRICS: summary + histogram ?? ?????? ???????
# =============================================================================

from pathlib import Path
import csv
import math
import statistics
from collections import Counter

submission_path = Path(SUBMISSION_PATH)
if not submission_path.exists():
    raise FileNotFoundError(f'submission file not found: {submission_path}')

lengths_all = []
solved_lengths = []
rows = []
with submission_path.open('r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        path = (row.get('path') or '').strip()
        length = 0 if path == '' else len(path.split('.'))
        rows.append(row)
        lengths_all.append(length)
        if length > 0:
            solved_lengths.append(length)

total_count = len(rows)
unsolved_count = total_count - len(solved_lengths)
solved_percent = 0.0 if total_count == 0 else 100.0 * len(solved_lengths) / total_count
total_len = sum(lengths_all)
mean_len_all = None if not lengths_all else statistics.mean(lengths_all)
median_len_all = None if not lengths_all else statistics.median(lengths_all)
max_len_solved = None if not solved_lengths else max(solved_lengths)
min_len_solved = None if not solved_lengths else min(solved_lengths)
mean_len_solved = None if not solved_lengths else statistics.mean(solved_lengths)
median_len_solved = None if not solved_lengths else statistics.median(solved_lengths)

summary = {
    'total_count': total_count,
    'unsolved_count': unsolved_count,
    'solved_percent': solved_percent,
    'total_len': total_len,
    'mean_len_all': mean_len_all,
    'median_len_all': median_len_all,
    'max_len_solved': max_len_solved,
    'min_len_solved': min_len_solved,
    'mean_len_solved': mean_len_solved,
    'median_len_solved': median_len_solved,
    'solved_lengths': solved_lengths,
}
print('METRICS_SUMMARY')
print(json.dumps(summary, ensure_ascii=False, indent=2))

hist = Counter(solved_lengths)
print('SOLVED_LENGTH_HISTOGRAM')
if not hist:
    print('(empty)')
else:
    max_count = max(hist.values())
    for length in sorted(hist):
        bar = '#' * max(1, round(50 * hist[length] / max_count))
        print(f'{length:4d} | {hist[length]:5d} | {bar}')


In [ ]:
# =============================================================================
# SUBMIT: ???????? submission.csv ? competition
# =============================================================================

submit_message = (
    f"test run: "
    f"beam={GLOBAL_BEAM_WIDTH} "
    f"depth={MAX_DEPTH} "
    f"samples={SAMPLE_START}..{SAMPLE_START + SAMPLE_COUNT - 1} "
    f"parallelism={INFERENCE_PARALLELISM} "
    f"b_micro={B_MICRO} "
    f"k_tile={K_EXPAND_TILE} "
    f"beta={BETA}"
)

print("SUBMIT_MESSAGE:", submit_message)

!kaggle competitions submit \
  -c cayley-py-megaminx \
  -f /kaggle/working/submission.csv \
  -m "$submit_message"
